# Chapter 11: Large Language Models & Transformers
## Notebook 02 — Pretrained LLMs

This notebook moves from "transformer math" to **using real pretrained models**: tokenizing with `AutoTokenizer`, extracting embeddings with `AutoModel`, building **frozen-embedding classifiers** with scikit-learn, and choosing between **BERT / RoBERTa / DistilBERT / GPT** for a task.

If `transformers` and `torch` are not installed, every cell falls back to a deterministic NumPy/sklearn stub that demonstrates the same shapes and APIs, so the notebook still runs end-to-end.

### What you'll learn

| Topic | Section |
|-------|--------|
| Loading a pretrained model | §2 |
| Tokenizing with `AutoTokenizer` | §3 |
| Generating contextual embeddings | §4 |
| Sentence vectors via mean pooling | §5 |
| Frozen-embedding classifier | §6 |
| Fine-tuning a classification head | §7 |
| Choosing BERT / RoBERTa / DistilBERT / GPT | §8 |

**Estimated time:** 3 hours

---
*Generated by Berta AI*

---
## 1. Setup

We'll try to import `transformers` and `torch`. If they're missing we use the chapter's `EmbeddingExtractor` fallback, which produces a deterministic hash-based embedding.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

HF_AVAILABLE = True
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    print("transformers + torch available — using real models.")
except ImportError as e:
    HF_AVAILABLE = False
    print(f"Hugging Face stack not available ({e}). Using fallback. "
          "Run `pip install torch transformers` to enable the real models.")

---
## 2. Loading a Pretrained Model

We use `distilbert-base-uncased` — a 6-layer, 66 M-parameter distillation of BERT that runs fine on CPU. Loading is just two calls.

In [ ]:
from config import MODEL_NAME

if HF_AVAILABLE:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
    model.eval()
    print(f"Loaded {MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")
else:
    from llm_utils import LLMTokenizerWrapper
    tokenizer = LLMTokenizerWrapper(MODEL_NAME)
    model = None
    print(f"Using fallback tokenizer for {MODEL_NAME}.")

---
## 3. Tokenization with `AutoTokenizer`

`AutoTokenizer` handles **special tokens** (`[CLS]`, `[SEP]`), **subword splitting**, and **truncation/padding** automatically.

In [ ]:
text = "Transformers are revolutionising natural language processing."

if HF_AVAILABLE:
    enc = tokenizer(text, return_tensors='pt', max_length=32, truncation=True, padding='max_length')
    print("input_ids:    ", enc['input_ids'][0].tolist())
    print("attention_mask:", enc['attention_mask'][0].tolist())
    print("tokens:       ", tokenizer.convert_ids_to_tokens(enc['input_ids'][0]))
else:
    ids = tokenizer.encode(text, max_length=16, padding=True)
    print("input_ids (fallback):", ids)
    print("tokens (fallback):    ", tokenizer.tokenize(text))

---
## 4. Generating Contextual Embeddings

Each token gets a vector; unlike word2vec, the **same word** in a different context gets a **different** vector — that's what "contextual" means.

In [ ]:
sentences = [
    "The bank approved my loan.",
    "We sat by the river bank.",
    "The pilot will land the plane soon.",
]

if HF_AVAILABLE:
    enc = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True, max_length=32)
    with torch.no_grad():
        out = model(**enc)
    last_hidden = out.last_hidden_state.numpy()  # (batch, seq, hidden)
    print("Last hidden state shape:", last_hidden.shape)
else:
    from llm_utils import EmbeddingExtractor
    extractor = EmbeddingExtractor(dim=64)
    last_hidden = np.stack([
        extractor.embed([s])[0:1].repeat(8, axis=0) for s in sentences
    ])
    print("Last hidden state shape (fallback):", last_hidden.shape)

---
## 5. Sentence Embeddings via Mean Pooling

`[CLS]` (BERT-style) or **mean pooling** of token vectors gives a single sentence vector. Mean pooling typically wins for similarity tasks.

In [ ]:
from llm_utils import mean_pool, cosine_sim, EmbeddingExtractor

if HF_AVAILABLE:
    mask = enc['attention_mask'].numpy()
    sent_vecs = mean_pool(last_hidden, mask=mask)
else:
    extractor = EmbeddingExtractor(dim=64)
    sent_vecs = extractor.embed(sentences)

sent_vecs /= np.clip(np.linalg.norm(sent_vecs, axis=1, keepdims=True), 1e-9, None)
print("Sentence vectors shape:", sent_vecs.shape)

sim = cosine_sim(sent_vecs, sent_vecs)
print("\nCosine similarity matrix:")
print(np.round(sim, 3))

In [ ]:
# Visualise sentences in 2-D with PCA.
from sklearn.decomposition import PCA

extractor = EmbeddingExtractor(dim=64)
demo_texts = [
    "The cat sat on the mat.",
    "A kitten rested on a rug.",
    "Stocks rose on the announcement.",
    "Equities climbed after the news.",
    "I love pizza.",
    "Pasta is my favourite food.",
]
demo_vecs = extractor.embed(demo_texts)
coords = PCA(n_components=2, random_state=42).fit_transform(demo_vecs)

plt.figure(figsize=(6, 4))
for (x, y), t in zip(coords, demo_texts):
    plt.scatter(x, y)
    plt.annotate(t[:25], (x, y), fontsize=8)
plt.title('Sentence embeddings (PCA)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout(); plt.show()

---
## 6. Frozen-Embedding Classifier

A surprisingly strong baseline: freeze the LLM, take sentence vectors, train a **logistic regression** on top. No fine-tuning, no GPU.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Tiny synthetic 2-class dataset (you'd use a real CSV in practice).
texts = [
    "The movie was fantastic and the acting was superb.",
    "I loved every minute of this film.",
    "Brilliant cinematography and a moving story.",
    "What a great experience at the theatre.",
    "Absolute waste of time, terrible plot.",
    "I hated the movie, the worst I have seen.",
    "Boring, slow and poorly acted.",
    "Do not watch, completely awful.",
]
labels = [1, 1, 1, 1, 0, 0, 0, 0]

extractor = EmbeddingExtractor(dim=64)
X = extractor.embed(texts)
X_tr, X_te, y_tr, y_te = train_test_split(X, labels, test_size=0.25, random_state=42, stratify=labels)

clf = LogisticRegression(max_iter=500, random_state=42).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
print("Accuracy:", accuracy_score(y_te, y_pred))
print(classification_report(y_te, y_pred, target_names=['neg', 'pos'], zero_division=0))

---
## 7. Sketch: Fine-Tuning a Classification Head

Frozen embeddings are great, but **fine-tuning** unfreezes the whole transformer (or just the top few layers) and updates them with task gradients. The pseudocode looks like:

```python
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args  = TrainingArguments(output_dir='out', num_train_epochs=3, per_device_train_batch_size=16)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val)
trainer.train()
```

Below we mimic the *shape* of fine-tuning by training a small MLP on top of the frozen embeddings.

In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42)
mlp.fit(X_tr, y_tr)
print("MLP-on-embeddings test acc:", mlp.score(X_te, y_te))
print("(Real fine-tuning would also update the transformer's parameters via Trainer.)")

---
## 8. Choosing a Pretrained Model

| Model | Params | Strengths | Use it for |
|-------|--------|-----------|-----------|
| **DistilBERT** | 66 M | Fast, 95% of BERT quality | Production classification, embeddings on CPU |
| **BERT-base** | 110 M | Strong general baseline | NER, classification, sentence-pair tasks |
| **RoBERTa-base** | 125 M | Better-trained BERT | Most encoder tasks where you need a step up |
| **MiniLM** | 22 M | Tiny, optimised for similarity | Semantic search, RAG retrieval |
| **GPT-2** | 124 M (small) | Open-weights decoder | Generation demos, perplexity studies |
| **Llama / Mistral** | 7 B+ | Strong instruction-tuned generators | Production chat, agents (with prompting/fine-tuning) |
| **T5 / BART** | 220 M+ | Encoder–decoder | Summarisation, translation, seq2seq tasks |

Rules of thumb:

- Need **vectors**? Reach for an encoder (DistilBERT or MiniLM).
- Need **generation**? Reach for a decoder (GPT-2 small for demos, Llama/Mistral for production).
- Need **structured input → structured output**? Reach for an encoder–decoder (T5, BART).

---
## 9. Key Takeaways

- `AutoTokenizer` and `AutoModel` are the two-line gateway into Hugging Face.
- **Mean-pool** the last hidden state to get a sentence vector — fast and strong.
- A **frozen-embedding + logistic regression** classifier is a great baseline before fine-tuning.
- Pick the model family that matches the task: encoder (vectors), decoder (generation), encoder–decoder (seq2seq).

Next: **Notebook 03** — generation strategies, KV cache, scaling laws, evaluation, and shipping LLM apps.

---
*Generated by Berta AI*